# Jackknife y Bootstrap: teoría y aplicación

Este cuaderno se enfoca únicamente en **jackknife** y **bootstrap**, conectando la teoría vista en clase con aplicaciones prácticas en datos tipo astroestadística.


## 1. Objetivos de aprendizaje

- Entender el principio de remuestreo para aproximar la distribución muestral de un estimador.
- Aplicar **Jackknife** para estimar sesgo, error estándar e influencia de observaciones.
- Aplicar **Bootstrap** para estimar incertidumbre e intervalos de confianza sin suponer normalidad estricta.
- Comparar ambos métodos sobre un mismo problema.


## 2. Teoría: Jackknife

Sea $\hat{\theta} = t(X_1,\dots,X_n)$ un estimador.

1. Se construyen los estimadores leave-one-out:
$$
\hat{\theta}_{(i)} = t(X_1,\dots,X_{i-1},X_{i+1},\dots,X_n),\quad i=1,\dots,n.
$$

2. Promedio jackknife:
$$
\bar{\theta}_{(\cdot)} = \frac{1}{n}\sum_{i=1}^n \hat{\theta}_{(i)}.
$$

3. Sesgo jackknife (aprox.):
$$
\widehat{\mathrm{bias}}_{JK}=(n-1)(\bar{\theta}_{(\cdot)}-\hat{\theta}).
$$

4. Error estándar jackknife:
$$
\widehat{SE}_{JK}=\sqrt{\frac{n-1}{n}\sum_{i=1}^n(\hat{\theta}_{(i)}-\bar{\theta}_{(\cdot)})^2}.
$$

**Idea clave:** mide estabilidad del estimador al quitar una observación.


## 3. Teoría: Bootstrap

El bootstrap no paramétrico reemplaza la distribución poblacional desconocida por la distribución empírica $\widehat{F}_n$.

1. Generar $B$ muestras con reemplazo, cada una de tamaño $n$.
2. Calcular $\hat{\theta}^{*(b)}$ en cada remuestra, $b=1,\dots,B$.
3. Usar la distribución de $\{\hat{\theta}^{*(b)}\}$ para:
   - estimar error estándar,
   - estimar sesgo,
   - construir intervalos de confianza (percentiles, básico, BCa, etc.).

En la práctica, el intervalo percentil al nivel $(1-\alpha)$ usa:
$$
\left[q_{\alpha/2}(\hat{\theta}^*),\,q_{1-\alpha/2}(\hat{\theta}^*)\right].
$$

**Idea clave:** aproxima la variabilidad real del estimador con cómputo intensivo.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)


## 4. Datos de ejemplo (estilo astroestadística)

Simulamos períodos orbitales positivos y asimétricos (en días), escenario donde el bootstrap suele ser muy útil.


In [ ]:
n = 80
periodos = rng.lognormal(mean=1.2, sigma=0.65, size=n)
log_periodos = np.log10(periodos)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(periodos, bins=18, color="#4C72B0", edgecolor="white")
ax[0].set_title("Período orbital (días)")
ax[0].set_xlabel("P")

ax[1].hist(log_periodos, bins=18, color="#55A868", edgecolor="white")
ax[1].set_title("log10(P)")
ax[1].set_xlabel("log10(P)")
plt.tight_layout()


## 5. Implementación de funciones de remuestreo


In [ ]:
def jackknife_stats(data, stat_func):
    x = np.asarray(data)
    n = len(x)
    theta_hat = stat_func(x)

    loo = np.empty(n, dtype=float)
    for i in range(n):
        loo[i] = stat_func(np.delete(x, i))

    loo_mean = loo.mean()
    bias = (n - 1) * (loo_mean - theta_hat)
    se = np.sqrt((n - 1) / n * np.sum((loo - loo_mean) ** 2))
    pseudovalues = n * theta_hat - (n - 1) * loo

    return {
        "theta_hat": theta_hat,
        "loo": loo,
        "loo_mean": loo_mean,
        "bias": bias,
        "se": se,
        "pseudovalues": pseudovalues,
    }


def bootstrap_stat(data, stat_func, B=6000, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    x = np.asarray(data)
    n = len(x)
    stats = np.empty(B, dtype=float)

    for b in range(B):
        sample = rng.choice(x, size=n, replace=True)
        stats[b] = stat_func(sample)

    return stats


def ci_percentil(boot_stats, alpha=0.05):
    low = np.quantile(boot_stats, alpha / 2)
    high = np.quantile(boot_stats, 1 - alpha / 2)
    return low, high


## 6. Aplicación 1: media y mediana de $\log_{10}(P)$

Calculamos estimaciones puntuales, sesgo y error estándar por ambos métodos.


In [ ]:
# Estadístico 1: media
jk_mean = jackknife_stats(log_periodos, np.mean)
boot_mean = bootstrap_stat(log_periodos, np.mean, B=8000, rng=rng)

# Estadístico 2: mediana
jk_median = jackknife_stats(log_periodos, np.median)
boot_median = bootstrap_stat(log_periodos, np.median, B=8000, rng=rng)

for name, jk, bt in [
    ("Media", jk_mean, boot_mean),
    ("Mediana", jk_median, boot_median),
]:
    theta = jk["theta_hat"]
    boot_bias = bt.mean() - theta
    boot_se = bt.std(ddof=1)
    ci = ci_percentil(bt, alpha=0.05)

    print(f"--- {name} de log10(P) ---")
    print(f"Estimación puntual: {theta:.4f}")
    print(f"Jackknife -> bias: {jk['bias']:.4f}, SE: {jk['se']:.4f}")
    print(f"Bootstrap -> bias: {boot_bias:.4f}, SE: {boot_se:.4f}")
    print(f"IC 95% bootstrap percentil: [{ci[0]:.4f}, {ci[1]:.4f}]\n")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(boot_mean, bins=35, color="#4C72B0", alpha=0.85, edgecolor="white")
ax[0].axvline(jk_mean["theta_hat"], color="black", ls="--", lw=2, label="Estimación original")
ax[0].set_title("Bootstrap de la media")
ax[0].set_xlabel("Media bootstrap")
ax[0].legend()

ax[1].hist(boot_median, bins=35, color="#C44E52", alpha=0.85, edgecolor="white")
ax[1].axvline(jk_median["theta_hat"], color="black", ls="--", lw=2, label="Estimación original")
ax[1].set_title("Bootstrap de la mediana")
ax[1].set_xlabel("Mediana bootstrap")
ax[1].legend()

plt.tight_layout()


## 7. Aplicación 2: influencia (Jackknife)

Visualizamos pseudovalores para detectar observaciones potencialmente influyentes.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(jk_median["loo"], marker="o", ms=4, lw=1)
ax[0].axhline(jk_median["loo_mean"], color="red", ls="--", label="Promedio LOO")
ax[0].set_title("Estimaciones leave-one-out (mediana)")
ax[0].set_xlabel("Índice eliminado")
ax[0].legend()

ax[1].hist(jk_median["pseudovalues"], bins=20, color="#8172B2", edgecolor="white")
ax[1].set_title("Pseudovalores jackknife")
ax[1].set_xlabel("Pseudovalor")

plt.tight_layout()


## 8. Conclusiones

- **Jackknife** es simple y útil para sesgo, error estándar e influencia punto a punto.
- **Bootstrap** ofrece una aproximación flexible de la distribución muestral y facilita intervalos de confianza sin supuestos paramétricos fuertes.
- En datos asimétricos (frecuentes en contextos astronómicos), el bootstrap suele ser más informativo para intervalos, mientras jackknife aporta diagnóstico de estabilidad.
